# FastMCP Demo
An MCP server built with **FastMCP** (the `fastmcp` Python package) for the tools,
with OpenAI function calling (`gpt-6-luna`) as the client.

*By Shivank Agarwal*

In [ ]:
!pip install -q openai requests fastmcp

In [ ]:
import getpass
import os

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

In [ ]:
import json
import requests
from openai import OpenAI
from fastmcp import FastMCP

openai_client = OpenAI()
MODEL = "gpt-6-luna"

# ---------------------------------------------------------------------------
# FastMCP server — tools are plain functions decorated with @mcp.tool()
# ---------------------------------------------------------------------------
mcp = FastMCP("MCP Demo Server")

WMO_CODES = {
    0: "Clear sky", 1: "Mainly clear", 2: "Partly cloudy", 3: "Overcast",
    45: "Fog", 48: "Icy fog",
    51: "Light drizzle", 53: "Moderate drizzle", 55: "Dense drizzle",
    61: "Slight rain", 63: "Moderate rain", 65: "Heavy rain",
    71: "Slight snow", 73: "Moderate snow", 75: "Heavy snow", 77: "Snow grains",
    80: "Slight showers", 81: "Moderate showers", 82: "Violent showers",
    85: "Slight snow showers", 86: "Heavy snow showers",
    95: "Thunderstorm", 96: "Thunderstorm with hail", 99: "Thunderstorm with heavy hail",
}


@mcp.tool()
def get_current_time(timezone: str = "UTC") -> str:
    """Get the current time in a given IANA timezone (e.g. 'Asia/Tokyo', 'Asia/Kolkata')."""
    from zoneinfo import ZoneInfo
    from datetime import datetime
    try:
        tz = ZoneInfo(timezone)
        now = datetime.now(tz)
        return f"Current time in {timezone}: {now.strftime('%Y-%m-%d %H:%M:%S %Z')}"
    except Exception as e:
        return f"Unknown timezone '{timezone}': {e}"


@mcp.tool()
def get_weather(city: str) -> str:
    """Get current weather for a city using Open-Meteo (no API key required)."""
    try:
        geo_resp = requests.get(
            "https://geocoding-api.open-meteo.com/v1/search",
            params={"name": city, "count": 1, "language": "en", "format": "json"},
            timeout=10,
        )
        geo_resp.raise_for_status()
        geo_data = geo_resp.json()
        if not geo_data.get("results"):
            return f"City '{city}' not found."
        loc = geo_data["results"][0]
        lat, lon = loc["latitude"], loc["longitude"]
        resolved_name = loc.get("name", city)

        wx_resp = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": lat,
                "longitude": lon,
                "current": "temperature_2m,relative_humidity_2m,apparent_temperature,wind_speed_10m,weather_code",
                "wind_speed_unit": "kmh",
                "timezone": "auto",
            },
            timeout=10,
        )
        wx_resp.raise_for_status()
        cur = wx_resp.json()["current"]
        desc = WMO_CODES.get(cur["weather_code"], f"Weather code {cur['weather_code']}")
        return (
            f"Weather in {resolved_name}: {desc}, "
            f"{cur['temperature_2m']}\u00b0C (feels like {cur['apparent_temperature']}\u00b0C), "
            f"Humidity: {cur['relative_humidity_2m']}%, Wind: {cur['wind_speed_10m']} km/h"
        )
    except Exception as e:
        return f"Could not get weather for {city}: {e}"


print("FastMCP server defined with tools: get_current_time, get_weather")

In [ ]:
from fastmcp import Client


async def chat_with_tools(user_message: str) -> str:
    """Send a message to OpenAI, dispatching tool calls through the FastMCP server."""
    async with Client(mcp) as client:
        tools = await client.list_tools()
        functions = [
            {
                "type": "function",
                "function": {
                    "name": t.name,
                    "description": t.description or "",
                    "parameters": t.input_schema,
                },
            }
            for t in tools
        ]

        messages = [{"role": "user", "content": user_message}]

        response = openai_client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=functions,
            tool_choice="auto",
            reasoning_effort="none",   # answer straight away, no hidden "thinking" tokens
        )
        message = response.choices[0].message

        if message.tool_calls:
            messages.append({
                "role": "assistant",
                "content": message.content,
                "tool_calls": [tc.model_dump() for tc in message.tool_calls],
            })

            for tool_call in message.tool_calls:
                tool_name = tool_call.function.name
                tool_args = json.loads(tool_call.function.arguments)

                result = await client.call_tool(tool_name, tool_args)
                text = result.content[0].text if result.content else ""

                print(f"  [Tool: {tool_name}({tool_args})] => {text}")
                messages.append({
                    "role": "tool",
                    "content": text,
                    "tool_call_id": tool_call.id,
                })

            final_response = openai_client.chat.completions.create(
                model=MODEL,
                messages=messages,
                reasoning_effort="none",
            )
            return final_response.choices[0].message.content

        return message.content


print("chat_with_tools defined.")

# **High-Level Flow of the Code**

**1. Connect to the MCP Server**

Create a client connection so the program can interact with tools exposed through the MCP server.

`async with Client(mcp) as client:`

**2. Discover Available Tools**

Fetch the list of tools from the MCP server (internally using the tools/list endpoint of Model Context Protocol).

`tools = await client.list_tools()`


**3. Convert MCP Tools to LLM Tool Format**

Transform the MCP tool metadata into the function-calling format expected by the OpenAI API.
```
functions = [
    {
        "type": "function",
        "function": {
            "name": t.name,
            "description": t.description,
            "parameters": t.input_schema,
        },
    }
    for t in tools
]

```


**4. Send User Message to the LLM**

Send the user message along with the available tools to the model (here `gpt-6-luna`), allowing it to decide whether to call a tool.



```
response = openai_client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=functions,
    tool_choice="auto",
    reasoning_effort="none",
)
```


**5. Check if the Model Requested a Tool**

Inspect the response to see whether the model decided to call a tool.

`if message.tool_calls:`

**6. Execute the Tool via MCP**

Call the requested tool on the MCP server with the arguments generated by the model.

`result = await client.call_tool(tool_name, tool_args)`

**7. Add Tool Output to Conversation**

Append the tool result to the message history so the model can use it for reasoning.



```
messages.append({
    "role": "tool",
    "content": text,
    "tool_call_id": tool_call.id,
})
```


**8. Generate the Final Response**

Send the updated conversation back to the model so it can produce the final answer.
```
final_response = openai_client.chat.completions.create(
    model=MODEL,
    messages=messages,
    reasoning_effort="none",
)
```
**9. Return the Final Answer**

Return the model’s final response to the user.

return final_response.choices[0].message.content

In [ ]:
async def run_demo():
    print("FastMCP Demo Starting...")

    demos = [
        "What time is it right now in Bangalore?",
        "What's the weather like in Bangalore?",
        "Can you tell me the time and weather in Tokyo?",
    ]

    for i, query in enumerate(demos, 1):
        print(f"\n--- Demo {i} ---")
        print(f"User: {query}")
        try:
            response = await chat_with_tools(query)
            print(f"Assistant: {response}")
        except Exception as e:
            print(f"Error: {e}")

    print("\nFastMCP Demo completed!")


await run_demo()